In [ ]:
# ============================================
# stack_experiment.ipynb
#
# [실험 목적]
# 임베딩 모델(bge-m3, KURE-v1, Qwen3-Embedding)을 바꿔가며, 실제
# 질문에서 정답 청크가 검색 순위 몇 등에 나오는지(find_rank) 정량
# 비교해서 어떤 임베딩이 우리 도메인(RFP 문서)에 가장 적합한지 확인하는 게 목적
#
# [진행 방식]
# - bge-m3, KURE-v1 임베딩을 각각 만들어 FAISS 인덱스 구축
# - find_rank() 함수로 실제 질문("한국철도공사 운행정보기록... 공동수급
#   지분율은?" 등)에서 정답 키워드가 담긴 청크가 몇 등으로 검색되는지 측정
# - Qwen3-Embedding-0.6B 모델도 추가로 로드해 동일한 테스트 케이스로
#   순위 비교
# - 새로운 실패 케이스("평택시 실시간 버스 도착정보...", "2025년도
#   행정정보시스템 위탁운영사업..." 등)를 추가로 발굴해 재검증
#
# [알아낸 것]
# 임베딩 모델별로 특정 질문에서 정답 청크의 검색 순위가 달라짐을 확인
# -> 이 결과가 이후 최종 임베딩 모델(KURE-v1) 채택의 근거 자료로 쓰임
# ============================================

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
import types
import pickle
import pandas as pd
from pathlib import Path

src_module = types.ModuleType('src')
chunking_module = types.ModuleType('src.chunking')

class Chunk:
    pass

chunking_module.Chunk = Chunk
src_module.chunking = chunking_module
sys.modules['src'] = src_module
sys.modules['src.chunking'] = chunking_module

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')

with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunk_objects = pickle.load(f)

with open(DATA_DIR / 'merged_docs.pkl', 'rb') as f:
    merged_df = pickle.load(f)

print(f"청크 개수: {len(chunk_objects)}")
print(f"문서 개수: {merged_df.shape}")
print(f"\n첫 청크 doc_id: {chunk_objects[0].doc_id}")
print(f"첫 청크 metadata: {chunk_objects[0].metadata}")
print(f"\nmerged_docs 파일명 예시: {merged_df['파일명'].iloc[0] if '파일명' in merged_df.columns else merged_df.columns.tolist()}")

청크 개수: 18239
문서 개수: (98, 34)

첫 청크 doc_id: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
첫 청크 metadata: {'발주_기관': '한영대학', '사업_금액': 130000000.0, 'budget_unknown': False, '입찰_참여_마감일': '2024-10-15 17:00:00', '입찰참여마감일_결측': False, '파일형식': 'hwp', 'doc_type': 'plain_text', 'source': 'raw_parsed'}

merged_docs 파일명 예시: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp


In [3]:
sfr_chunks = [c for c in chunk_objects if '국방과학연구소_기록관리시스템' in c.doc_id]
print(f"국방과학연구소 청크 개수: {len(sfr_chunks)}")

has_table_marker = any('[표]' in c.text for c in sfr_chunks)
has_sfr = any('SFR-001' in c.text for c in sfr_chunks)
print(f"[표] 마커 포함 청크 존재: {has_table_marker}")
print(f"SFR-001 포함 청크 존재: {has_sfr}")

print("\n첫 청크 doc_type 확인:")
doc_types = set(c.metadata.get('doc_type') for c in chunk_objects[:100])
print(doc_types)

국방과학연구소 청크 개수: 139
[표] 마커 포함 청크 존재: True
SFR-001 포함 청크 존재: True

첫 청크 doc_type 확인:
{'plain_text'}


In [4]:
row = merged_df[merged_df['파일명'] == '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp'].iloc[0]

print("merged_docs 값")
for col in merged_df.columns:
    if '금액' in col or '발주' in col or '마감' in col:
        print(f"{col}: {row[col]}")

print("\n청크 metadata 값")
print(chunk_objects[0].metadata)

merged_docs 값
사업 금액: 130000000.0
발주 기관: 한영대학
입찰 참여 마감일: 2024-10-15 17:00:00
사업_금액_정제: 130000000.0
사업_금액_출처: csv
사업_금액_후보텍스트: None
입찰 참여 마감일_dt: 2024-10-15 17:00:00
입찰참여마감일_결측: False
입찰참여마감일_정제: 2024-10-15 17:00:00
입찰참여마감일_출처: csv
입찰참여마감일_후보텍스트: None

청크 metadata 값
{'발주_기관': '한영대학', '사업_금액': 130000000.0, 'budget_unknown': False, '입찰_참여_마감일': '2024-10-15 17:00:00', '입찰참여마감일_결측': False, '파일형식': 'hwp', 'doc_type': 'plain_text', 'source': 'raw_parsed'}


In [5]:
all_chunks_final = []
chunk_metadata_final = []

for c in chunk_objects:
    all_chunks_final.append(c.text)
    meta_info = c.metadata if c.metadata else {}
    chunk_metadata_final.append({
        '파일명': c.doc_id,
        '발주기관': meta_info.get('발주_기관', ''),
        '사업금액': meta_info.get('사업_금액', None),
        '마감일': meta_info.get('입찰_참여_마감일', ''),
    })

print(f"변환 완료: {len(all_chunks_final)}개")
print(chunk_metadata_final[0])

변환 완료: 18239개
{'파일명': '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp', '발주기관': '한영대학', '사업금액': 130000000.0, '마감일': '2024-10-15 17:00:00'}


In [6]:
import torch
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

CUDA 사용 가능: True


In [9]:
from FlagEmbedding import BGEM3FlagModel
bge_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

bge_embeddings = bge_model.encode(all_chunks_final, batch_size=12)['dense_vecs']
print(f"bge-m3 임베딩 shape: {bge_embeddings.shape}")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 1520/1520 [03:29<00:00,  7.25it/s]


bge-m3 임베딩 shape: (18239, 1024)


In [24]:
from sentence_transformers import SentenceTransformer
kure_model = SentenceTransformer('nlpai-lab/KURE-v1')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [23]:
kure_embeddings = kure_model.encode(all_chunks_final, batch_size=64, show_progress_bar=True)
print(f"KURE 임베딩 shape: {kure_embeddings.shape}")

Batches:   0%|          | 0/285 [00:00<?, ?it/s]

KURE 임베딩 shape: (18239, 1024)


In [25]:
import faiss
import numpy as np

dim_bge = bge_embeddings.shape[1]
index_bge = faiss.IndexFlatL2(dim_bge)
index_bge.add(np.array(bge_embeddings).astype('float32'))

dim_kure = kure_embeddings.shape[1]
index_kure = faiss.IndexFlatL2(dim_kure)
index_kure.add(np.array(kure_embeddings).astype('float32'))

print(f"bge 인덱스: {index_bge.ntotal}개, KURE 인덱스: {index_kure.ntotal}개")

bge 인덱스: 18239개, KURE 인덱스: 18239개


In [26]:
def find_rank(query, target_keyword, target_doc_keyword, model, index, all_chunks, chunk_metadata):
    query_embedding = model.encode([query])
    if isinstance(query_embedding, dict):
        query_embedding = query_embedding['dense_vecs']
    distances, indices = index.search(np.array(query_embedding).astype('float32'), len(all_chunks))

    for rank, idx in enumerate(indices[0], 1):
        if target_doc_keyword in chunk_metadata[idx]['파일명'] and target_keyword in all_chunks[idx]:
            return rank
    return None

In [27]:
test_cases = [
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 시 최소지분율과 구성원 수 상한은 어떻게 되나요?", "지분율", "운행정보기록"),
    ("울산광역시 버스정보시스템 확대 구축 및 기능개선 사업에서 하도급이 가능한가요?", "하도급", "울산광역시"),
    ("검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업 예산은 얼마인가요?", "사 업 비", "대검찰청"),
]

for query, keyword, doc_keyword in test_cases:
    bge_rank = find_rank(query, keyword, doc_keyword, bge_model, index_bge, all_chunks_final, chunk_metadata_final)
    kure_rank = find_rank(query, keyword, doc_keyword, kure_model, index_kure, all_chunks_final, chunk_metadata_final)
    print(f"질문: {query[:40]}...")
    print(f"  bge-m3 순위: {bge_rank}")
    print(f"  KURE 순위: {kure_rank}")
    print()

질문: 한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 시 최소지...
  bge-m3 순위: 3387
  KURE 순위: 384

질문: 울산광역시 버스정보시스템 확대 구축 및 기능개선 사업에서 하도급이 가능한...
  bge-m3 순위: 3
  KURE 순위: 7

질문: 검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업 예산은 얼마인가요?...
  bge-m3 순위: 1
  KURE 순위: 1



In [28]:
test_cases_all = [
    ("검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업과 국가 교육과정 정보 제공 사이트 운영 사업 중 예산이 더 큰 쪽은 어디이며 차액은 얼마인가요?", "사 업 비", "대검찰청"),
    ("한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제출물의 수량은 어떻게 되나요?", "10부", "한영대학"),
    ("'2024년 버스정보시스템 확대 구축 및 기능개선 용역'의 사업예산은 얼마인가요?", "986,945,000", "울산광역시"),
    ("인천공항운영서비스가 경영업무를 하나로 통합하는 차세대 시스템에 투입하는 금액은 얼마이며, 부가가치세는 별도인가요?", "996,356,000", "인천공항운영서비스"),
    ("한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성한 예산은 얼마인가요?", "181,913,000", "네팔 수자원관리"),
    ("한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?", "6개월", "네팔 수자원관리"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?", "하도급을 불허", "운행정보기록"),
    ("수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입찰이 재입찰 또는 재공고입찰로 진행되면 최초 조건을 변경할 수 있나요?", "재입찰", "수협중앙회"),
    ("수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업의 계약보증금 기준은 무엇인가요?", "계약보증금", "수협중앙회"),
    ("'2025년 통합접수시스템 운영'의 기술평가와 가격평가 비중은 어떻게 되며, 기술평가 내부 배점은 어떻게 나뉘나요?", "기술평가", "경기도일자리재단"),
    ("울산광역시 버스정보시스템 확대 구축 및 기능개선 사업에서 하도급이 가능한가요? 가능하다면 핵심 제한은 무엇인가요?", "하도급", "울산광역시"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?", "제안서 보상", "운행정보기록"),
    ("국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 할 유지관리·교육 의무", "하자보수", "국립인천해양박물관"),
    ("부산관광공사 경영정보시스템 기능개선 사업에서 낙찰자가 협상 성립 후 10일 이내 계약을 체결하지 않으면 어떤 불이익이 있나요?", "부정당업자", "부산관광공사"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 시 최소지분율과 구성원 수 상한은 어떻게 되나요?", "지분율", "운행정보기록"),
    ("철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역의 평가 배점은?", "기술", "국가철도공단"),
]

results = []
for query, keyword, doc_keyword in test_cases_all:
    bge_rank = find_rank(query, keyword, doc_keyword, bge_model, index_bge, all_chunks_final, chunk_metadata_final)
    kure_rank = find_rank(query, keyword, doc_keyword, kure_model, index_kure, all_chunks_final, chunk_metadata_final)
    results.append((query[:30], bge_rank, kure_rank))
    print(f"{query[:30]}... | bge: {bge_rank} | KURE: {kure_rank}")

검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업과 국... | bge: 1 | KURE: 1
한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안... | bge: 9 | KURE: 5
'2024년 버스정보시스템 확대 구축 및 기능개선 용역... | bge: 1 | KURE: 2
인천공항운영서비스가 경영업무를 하나로 통합하는 차세대 ... | bge: 2 | KURE: 2
한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스... | bge: 1 | KURE: 2
한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 ... | bge: 1 | KURE: 2
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하... | bge: 2842 | KURE: 1252
수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수... | bge: 4 | KURE: 5
수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수... | bge: 55 | KURE: 20
'2025년 통합접수시스템 운영'의 기술평가와 가격평가... | bge: 10 | KURE: 13
울산광역시 버스정보시스템 확대 구축 및 기능개선 사업에... | bge: 4 | KURE: 4
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜... | bge: 313 | KURE: 87
국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 ... | bge: 350 | KURE: 151
부산관광공사 경영정보시스템 기능개선 사업에서 낙찰자가 ... | bge: 5 | KURE: 2
한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 ... | bge: 3387 | KURE: 384
철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역... | bge: 1 | KURE: 1


In [29]:
qwen_model = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B', model_kwargs={'torch_dtype': torch.float16})
print(qwen_model.device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

cuda:0


In [9]:
import pickle

with open(DATA_DIR / 'bge_embeddings.pkl', 'rb') as f:
    bge_embeddings = pickle.load(f)

with open(DATA_DIR / 'kure_embeddings.pkl', 'rb') as f:
    kure_embeddings = pickle.load(f)

print(f"bge shape: {bge_embeddings.shape}")
print(f"kure shape: {kure_embeddings.shape}")

bge shape: (18239, 1024)
kure shape: (18239, 1024)


In [5]:
import sys
import types
import pickle
from pathlib import Path

src_module = types.ModuleType('src')
chunking_module = types.ModuleType('src.chunking')

class Chunk:
    pass

chunking_module.Chunk = Chunk
src_module.chunking = chunking_module
sys.modules['src'] = src_module
sys.modules['src.chunking'] = chunking_module

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')

with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunk_objects = pickle.load(f)

all_chunks_final = [c.text for c in chunk_objects]
print(f"청크 로드 완료: {len(all_chunks_final)}개")

청크 로드 완료: 18239개


In [6]:
import torch
from sentence_transformers import SentenceTransformer

qwen_model = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B', model_kwargs={'torch_dtype': torch.float16})
print(qwen_model.device)
print(f"시작 시 GPU 메모리: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

cuda:0
시작 시 GPU 메모리: 1.19 GB


In [8]:
qwen_embeddings = qwen_model.encode(all_chunks_final, batch_size=32, show_progress_bar=True)
print(f"Qwen 임베딩 shape: {qwen_embeddings.shape}")

Batches:   0%|          | 0/570 [00:00<?, ?it/s]

Qwen 임베딩 shape: (18239, 1024)


In [9]:
import pickle
with open(DATA_DIR / 'qwen_embeddings.pkl', 'wb') as f:
    pickle.dump(qwen_embeddings, f)
print("Qwen 임베딩 저장 완료")

Qwen 임베딩 저장 완료


In [10]:
dim_qwen = qwen_embeddings.shape[1]
index_qwen = faiss.IndexFlatL2(dim_qwen)
index_qwen.add(np.array(qwen_embeddings).astype('float32'))

print(f"Qwen 인덱스: {index_qwen.ntotal}개")

Qwen 인덱스: 18239개


In [12]:
def find_rank(query, target_keyword, target_doc_keyword, model, index, all_chunks, chunk_metadata):
    query_embedding = model.encode([query])
    if isinstance(query_embedding, dict):
        query_embedding = query_embedding['dense_vecs']
    distances, indices = index.search(np.array(query_embedding).astype('float32'), len(all_chunks))

    for rank, idx in enumerate(indices[0], 1):
        if target_doc_keyword in chunk_metadata[idx]['파일명'] and target_keyword in all_chunks[idx]:
            return rank
    return None

In [15]:
chunk_metadata_final = []
for c in chunk_objects:
    meta_info = c.metadata if c.metadata else {}
    chunk_metadata_final.append({
        '파일명': c.doc_id,
        '발주기관': meta_info.get('발주_기관', ''),
        '사업금액': meta_info.get('사업_금액', None),
        '마감일': meta_info.get('입찰_참여_마감일', ''),
    })

print(f"복구 완료: {len(chunk_metadata_final)}개")

복구 완료: 18239개


In [16]:
test_cases_all = [
    ("검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업과 국가 교육과정 정보 제공 사이트 운영 사업 중 예산이 더 큰 쪽은 어디이며 차액은 얼마인가요?", "사 업 비", "대검찰청"),
    ("한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안 관련 제출물의 수량은 어떻게 되나요?", "10부", "한영대학"),
    ("'2024년 버스정보시스템 확대 구축 및 기능개선 용역'의 사업예산은 얼마인가요?", "986,945,000", "울산광역시"),
    ("인천공항운영서비스가 경영업무를 하나로 통합하는 차세대 시스템에 투입하는 금액은 얼마이며, 부가가치세는 별도인가요?", "996,356,000", "인천공항운영서비스"),
    ("한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성한 예산은 얼마인가요?", "181,913,000", "네팔 수자원관리"),
    ("한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?", "6개월", "네팔 수자원관리"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하도급이 가능한가요?", "하도급을 불허", "운행정보기록"),
    ("수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입찰이 재입찰 또는 재공고입찰로 진행되면 최초 조건을 변경할 수 있나요?", "재입찰", "수협중앙회"),
    ("수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 사업의 계약보증금 기준은 무엇인가요?", "계약보증금", "수협중앙회"),
    ("'2025년 통합접수시스템 운영'의 기술평가와 가격평가 비중은 어떻게 되며, 기술평가 내부 배점은 어떻게 나뉘나요?", "기술평가", "경기도일자리재단"),
    ("울산광역시 버스정보시스템 확대 구축 및 기능개선 사업에서 하도급이 가능한가요? 가능하다면 핵심 제한은 무엇인가요?", "하도급", "울산광역시"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜 제안서 보상 대상이 아닌가요?", "제안서 보상", "운행정보기록"),
    ("국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 부담해야 할 유지관리·교육 의무", "하자보수", "국립인천해양박물관"),
    ("부산관광공사 경영정보시스템 기능개선 사업에서 낙찰자가 협상 성립 후 10일 이내 계약을 체결하지 않으면 어떤 불이익이 있나요?", "부정당업자", "부산관광공사"),
    ("한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 공동수급 시 최소지분율과 구성원 수 상한은 어떻게 되나요?", "지분율", "운행정보기록"),
    ("철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역의 평가 배점은?", "기술", "국가철도공단"),
]

for query, keyword, doc_keyword in test_cases_all:
    qwen_rank = find_rank(query, keyword, doc_keyword, qwen_model, index_qwen, all_chunks_final, chunk_metadata_final)
    print(f"{query[:30]}... | Qwen 순위: {qwen_rank}")

검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업과 국... | Qwen 순위: 1
한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안... | Qwen 순위: 9
'2024년 버스정보시스템 확대 구축 및 기능개선 용역... | Qwen 순위: 1
인천공항운영서비스가 경영업무를 하나로 통합하는 차세대 ... | Qwen 순위: 1
한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스... | Qwen 순위: 1
한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 ... | Qwen 순위: 3
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하... | Qwen 순위: 5009
수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수... | Qwen 순위: 5
수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수... | Qwen 순위: 17
'2025년 통합접수시스템 운영'의 기술평가와 가격평가... | Qwen 순위: 9
울산광역시 버스정보시스템 확대 구축 및 기능개선 사업에... | Qwen 순위: 1
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜... | Qwen 순위: 369
국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 ... | Qwen 순위: 39
부산관광공사 경영정보시스템 기능개선 사업에서 낙찰자가 ... | Qwen 순위: 14
한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 ... | Qwen 순위: 1775
철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역... | Qwen 순위: 2


In [17]:
for query, keyword, doc_keyword in test_cases_all:
    qwen_rank = find_rank(query, keyword, doc_keyword, qwen_model, index_qwen, all_chunks_final, chunk_metadata_final)
    print(f"{query[:30]}... | Qwen 순위: {qwen_rank}")

검찰의 아태지역 사이버범죄 교육 플랫폼 개선 사업과 국... | Qwen 순위: 1
한영대학교 트랙운영 학사정보시스템 고도화 사업에서 제안... | Qwen 순위: 9
'2024년 버스정보시스템 확대 구축 및 기능개선 용역... | Qwen 순위: 1
인천공항운영서비스가 경영업무를 하나로 통합하는 차세대 ... | Qwen 순위: 1
한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스... | Qwen 순위: 1
한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 ... | Qwen 순위: 3
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 하... | Qwen 순위: 5009
수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수... | Qwen 순위: 5
수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수... | Qwen 순위: 17
'2025년 통합접수시스템 운영'의 기술평가와 가격평가... | Qwen 순위: 9
울산광역시 버스정보시스템 확대 구축 및 기능개선 사업에... | Qwen 순위: 1
한국철도공사 운행정보기록 자동분석시스템 개량 사업은 왜... | Qwen 순위: 369
국립인천해양박물관 해양자료관리시스템 구축사가 검수 후 ... | Qwen 순위: 39
부산관광공사 경영정보시스템 기능개선 사업에서 낙찰자가 ... | Qwen 순위: 14
한국철도공사 운행정보기록 자동분석시스템 개량 사업에서 ... | Qwen 순위: 1775
철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역... | Qwen 순위: 2


In [21]:
from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import SentenceTransformer

bge_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
kure_model = SentenceTransformer('nlpai-lab/KURE-v1', model_kwargs={'torch_dtype': torch.float16})

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [22]:
with open(DATA_DIR / 'bge_embeddings.pkl', 'rb') as f:
    bge_embeddings = pickle.load(f)
with open(DATA_DIR / 'kure_embeddings.pkl', 'rb') as f:
    kure_embeddings = pickle.load(f)

index_bge = faiss.IndexFlatL2(bge_embeddings.shape[1])
index_bge.add(np.array(bge_embeddings).astype('float32'))

index_kure = faiss.IndexFlatL2(kure_embeddings.shape[1])
index_kure.add(np.array(kure_embeddings).astype('float32'))

print(f"bge: {index_bge.ntotal}, KURE: {index_kure.ntotal}")

bge: 18239, KURE: 18239


In [23]:
new_test_cases = [
    ("평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?", "2024. 10. 31", "평택시"),
    ("2025년도 행정정보시스템 위탁운영사업은 어떤 계약 방식으로 체결되며 예산은 얼마인가요?", "103", "중앙선거관리위원회"),
    ("자본금 5억원이고 주된 영업소가 서울에 있는 중소 소프트웨어사업자가 부산관광공사 경영정보시스템 기능개선 입찰에 참여할 수 있나요?", "부산", "부산관광공사"),
    ("서울 디지털성범죄 안심지원센터 통합 사업에서 AI 삭제지원 관련 주요 구축 범위", "AI 삭제지원", "서울특별시 여성가족재단"),
    ("국방과학연구소 '기록관리시스템 통합 활용 및 보안 환경 구축' 사업에 공동수급체 소속이 아닌 외부 인력을 투입해도 되나요? 투입할 경우 하도급으로 처리되나요?", "하도급", "국방과학연구소_기록관리시스템"),
    ("부산관광공사의 '경영정보시스템 기능개선'은 공사·물품·용역 중 어떤 유형인가요?", "용역", "부산관광공사"),
    ("부산국제영화제가 온라인 행사 서비스를 다시 개발하고 운영 지원을 받는 용역에 편성한 금액은 얼마인가요? 세금 포함 여부도 알려주세요.", "243,000,000", "부산국제영화제"),
]

for query, keyword, doc_keyword in new_test_cases:
    bge_rank = find_rank(query, keyword, doc_keyword, bge_model, index_bge, all_chunks_final, chunk_metadata_final)
    kure_rank = find_rank(query, keyword, doc_keyword, kure_model, index_kure, all_chunks_final, chunk_metadata_final)
    qwen_rank = find_rank(query, keyword, doc_keyword, qwen_model, index_qwen, all_chunks_final, chunk_metadata_final)
    print(f"{query[:30]}... | bge: {bge_rank} | KURE: {kure_rank} | Qwen: {qwen_rank}")

평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는... | bge: 2 | KURE: 2 | Qwen: 4
2025년도 행정정보시스템 위탁운영사업은 어떤 계약 방... | bge: 1 | KURE: 1 | Qwen: 1
자본금 5억원이고 주된 영업소가 서울에 있는 중소 소프... | bge: 1 | KURE: 1 | Qwen: 1
서울 디지털성범죄 안심지원센터 통합 사업에서 AI 삭제... | bge: 2 | KURE: 2 | Qwen: 2
국방과학연구소 '기록관리시스템 통합 활용 및 보안 환경... | bge: 75 | KURE: 4 | Qwen: 75
부산관광공사의 '경영정보시스템 기능개선'은 공사·물품·... | bge: 1 | KURE: 1 | Qwen: 1
부산국제영화제가 온라인 행사 서비스를 다시 개발하고 운... | bge: 17 | KURE: 3 | Qwen: 7
